# Data Quality Checks (4 Standard dbt Tests)

**Purpose**: Run the 4 standard dbt data quality tests on any gold table

## The 4 Tests

1. **Unique** - No duplicate values in key columns (e.g., customer_number, product_number)
2. **Not Null** - Required columns have no NULL values (e.g., customer_id, product_name)
3. **Accepted Values** - Columns only contain approved values (e.g., gender in ['Male', 'Female'])
4. **Relationships** - Foreign keys exist in referenced tables (e.g., customer_key in fact_sales exists in dim_customers)

## Usage

1. **Run Cell 2** to auto-discover available gold tables (dynamically queries `workspace.gold` schema)
2. Select `table_name` parameter from dropdown (auto-populated with all `dim_*` and `fact_*` tables)
3. Run remaining cells to execute quality checks
4. Review results:
   - Pass/Fail summary
   - Failed records quarantined
   - Metrics logged to `gold.data_quality_metrics`

## Dynamic Parameter

The `table_name` parameter is **dynamically populated** by querying the gold schema. When you add new dimension or fact tables, just re-run Cell 2 and they'll appear in the dropdown automatically.

## Quarantine Pattern

Failed records are moved to `{table_name}_quarantine` for investigation and remediation.

In [0]:
# Dynamically discover all gold dimension and fact tables
tables_df = spark.sql("""
    SHOW TABLES IN workspace.gold
""").filter(
    "tableName LIKE 'dim_%' OR tableName LIKE 'fact_%'"
).select('tableName')

# Convert to pandas and get list
available_tables = tables_df.toPandas()['tableName'].tolist()

# Filter out quarantine and metrics tables
available_tables = [t for t in available_tables if not t.endswith('_quarantine') and t != 'data_quality_metrics']
available_tables = sorted(available_tables)  # Alphabetical order

if not available_tables:
    raise ValueError("No dim_* or fact_* tables found in workspace.gold schema")

print(f"📊 Found {len(available_tables)} gold tables:")
for table in available_tables:
    print(f"   - {table}")

# Create/update the parameter widget with discovered tables
try:
    dbutils.widgets.remove('table_name')
except:
    pass  # Widget doesn't exist yet

dbutils.widgets.dropdown('table_name', available_tables[0], available_tables, 'Table Name')
print(f"\n✅ Parameter 'table_name' created with {len(available_tables)} choices")
print(f"   Default: {available_tables[0]}")

In [0]:
# Get table_name parameter (with default fallback)
try:
    table_name = dbutils.widgets.get("table_name")
    if not table_name or table_name.strip() == '':
        table_name = "dim_customers"  # Default if empty
except:
    table_name = "dim_customers"  # Default if widget doesn't exist

# Define the 4 standard dbt tests for each table
table_tests = {
    "dim_customers": {
        "table": "workspace.gold.dim_customers",
        "quarantine": "workspace.gold.dim_customers_quarantine",
        "filter": "is_current = TRUE",  # Only check current SCD records
        
        # Test 1: Unique - these columns should have no duplicates
        "unique": ["customer_number"],
        
        # Test 2: Not Null - these columns must not be NULL
        "not_null": ["customer_number", "customer_id", "first_name", "last_name"],
        
        # Test 3: Accepted Values - columns with specific allowed values
        "accepted_values": {
            "gender": ["Male", "Female", "n/a"],
            "marital_status": ["Single", "Married", "n/a"]
        },
        
        # Test 4: Relationships - no foreign keys for dimensions
        "relationships": []
    },
    
    "dim_products": {
        "table": "workspace.gold.dim_products",
        "quarantine": "workspace.gold.dim_products_quarantine",
        "filter": "is_current = TRUE",
        
        "unique": ["product_number"],
        "not_null": ["product_number", "product_id", "product_name"],
        "accepted_values": {},
        "relationships": []
    },
    
    "fact_sales": {
        "table": "workspace.gold.fact_sales",
        "quarantine": "workspace.gold.fact_sales_quarantine",
        "filter": None,  # No filter for facts
        
        "unique": [],  # Facts typically don't have unique constraints
        "not_null": ["customer_key", "product_key", "order_date", "quantity"],
        "accepted_values": {},
        
        # Test 4: Relationships - foreign keys must exist
        "relationships": [
            {"column": "customer_key", "references": "workspace.gold.dim_customers", "ref_column": "customer_key", "ref_filter": "is_current = TRUE"},
            {"column": "product_key", "references": "workspace.gold.dim_products", "ref_column": "product_key", "ref_filter": "is_current = TRUE"}
        ]
    },
    
    "fact_sales_enriched": {
        "table": "workspace.gold.fact_sales_enriched",
        "quarantine": "workspace.gold.fact_sales_enriched_quarantine",
        "filter": None,  # No filter for facts
        
        "unique": [],  # Facts typically don't have unique constraints
        "not_null": ["customer_key", "product_key", "order_date", "quantity"],
        "accepted_values": {},
        
        # Test 4: Relationships - foreign keys must exist
        "relationships": [
            {"column": "customer_key", "references": "workspace.gold.dim_customers", "ref_column": "customer_key", "ref_filter": "is_current = TRUE"},
            {"column": "product_key", "references": "workspace.gold.dim_products", "ref_column": "product_key", "ref_filter": "is_current = TRUE"}
        ]
    }
}

config = table_tests.get(table_name)
if not config:
    raise ValueError(f"Unknown table: {table_name}. Available: {list(table_tests.keys())}")

print(f"✅ Running 4 dbt tests on: {config['table']}")
print(f"   1. Unique: {len(config['unique'])} columns")
print(f"   2. Not Null: {len(config['not_null'])} columns")
print(f"   3. Accepted Values: {len(config['accepted_values'])} columns")
print(f"   4. Relationships: {len(config['relationships'])} foreign keys")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read table with optional filter
df = spark.table(config['table'])
if config['filter']:
    df = df.filter(config['filter'])

total_records = df.count()
failed_records = []

print(f"\n📋 Checking {total_records} records...\n")

# ==================== TEST 1: UNIQUE ====================
print("1️⃣  UNIQUE test")
for col in config['unique']:
    if col in df.columns:
        duplicates = df.groupBy(col).count().filter(F.col('count') > 1)
        dup_count = duplicates.count()
        if dup_count > 0:
            print(f"   ❌ {col}: {dup_count} duplicate values found")
            # Get records with duplicates
            dup_values = duplicates.select(col).toPandas()[col].tolist()
            dup_df = df.filter(F.col(col).isin(dup_values)) \
                .withColumn('test_failed', F.lit('unique')) \
                .withColumn('column_name', F.lit(col)) \
                .withColumn('failure_reason', F.lit(f"Duplicate value in {col}"))
            failed_records.append(dup_df)
        else:
            print(f"   ✅ {col}: All unique")

# ==================== TEST 2: NOT NULL ====================
print("\n2️⃣  NOT NULL test")
for col in config['not_null']:
    if col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            print(f"   ❌ {col}: {null_count} NULL values found")
            null_df = df.filter(F.col(col).isNull()) \
                .withColumn('test_failed', F.lit('not_null')) \
                .withColumn('column_name', F.lit(col)) \
                .withColumn('failure_reason', F.lit(f"NULL value in {col}"))
            failed_records.append(null_df)
        else:
            print(f"   ✅ {col}: No NULLs")

# ==================== TEST 3: ACCEPTED VALUES ====================
print("\n3️⃣  ACCEPTED VALUES test")
for col, valid_values in config['accepted_values'].items():
    if col in df.columns:
        invalid = df.filter(~F.col(col).isin(valid_values) & F.col(col).isNotNull())
        invalid_count = invalid.count()
        if invalid_count > 0:
            print(f"   ❌ {col}: {invalid_count} invalid values (allowed: {valid_values})")
            invalid_df = invalid \
                .withColumn('test_failed', F.lit('accepted_values')) \
                .withColumn('column_name', F.lit(col)) \
                .withColumn('failure_reason', F.concat(F.lit(f"Invalid value in {col}: "), F.col(col).cast('string')))
            failed_records.append(invalid_df)
        else:
            print(f"   ✅ {col}: All values valid")

# ==================== TEST 4: RELATIONSHIPS ====================
print("\n4️⃣  RELATIONSHIPS test")
for rel in config['relationships']:
    col = rel['column']
    ref_table = rel['references']
    ref_col = rel['ref_column']
    ref_filter = rel.get('ref_filter')
    
    if col in df.columns:
        # Get valid values from reference table
        ref_df = spark.table(ref_table)
        if ref_filter:
            ref_df = ref_df.filter(ref_filter)
        valid_keys = ref_df.select(ref_col).distinct()
        
        # Find orphaned records (foreign key doesn't exist)
        orphans = df.join(valid_keys, df[col] == valid_keys[ref_col], 'left_anti')
        orphan_count = orphans.count()
        
        if orphan_count > 0:
            print(f"   ❌ {col}: {orphan_count} orphaned records (no match in {ref_table}.{ref_col})")
            orphan_df = orphans \
                .withColumn('test_failed', F.lit('relationships')) \
                .withColumn('column_name', F.lit(col)) \
                .withColumn('failure_reason', F.lit(f"Foreign key {col} not found in {ref_table}.{ref_col}"))
            failed_records.append(orphan_df)
        else:
            print(f"   ✅ {col}: All foreign keys valid")

# Combine all failures
if failed_records:
    all_failures = failed_records[0]
    for fail_df in failed_records[1:]:
        all_failures = all_failures.unionByName(fail_df, allowMissingColumns=True)
    all_failures.createOrReplaceTempView('failed_records')
    total_failures = all_failures.count()
    print(f"\n❌ Total failed records: {total_failures}")
    print(f"\n📊 Pass rate: {round((total_records - total_failures) / total_records * 100, 2)}%")
else:
    # Create empty DataFrame with correct schema (not df.schema!)
    from pyspark.sql.types import StructType, StructField, StringType
    
    # Build schema with all source columns plus test metadata columns
    failure_schema = df.schema
    failure_schema = failure_schema.add(StructField('test_failed', StringType(), False))
    failure_schema = failure_schema.add(StructField('column_name', StringType(), False))
    failure_schema = failure_schema.add(StructField('failure_reason', StringType(), False))
    
    spark.createDataFrame([], failure_schema).createOrReplaceTempView('failed_records')
    print(f"\n✅ All tests passed!")
    print(f"\n📊 Pass rate: 100.0%")

In [0]:
%sql
-- Show summary of test failures
SELECT 
  test_failed as test,
  column_name,
  COUNT(*) as failed_count
FROM failed_records
GROUP BY test_failed, column_name
ORDER BY 
  CASE test_failed
    WHEN 'unique' THEN 1
    WHEN 'not_null' THEN 2
    WHEN 'accepted_values' THEN 3
    WHEN 'relationships' THEN 4
  END,
  failed_count DESC;

In [0]:
# Log test results to metrics table (using Python for dynamic table names)
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, DoubleType
from datetime import datetime

# Count failures by test type from failed_records temp view
failed_df = spark.table('failed_records')
failed_count = failed_df.count()

if failed_count > 0:
    unique_failures = failed_df.filter(F.col('test_failed') == 'unique').count()
    not_null_failures = failed_df.filter(F.col('test_failed') == 'not_null').count()
    accepted_failures = failed_df.filter(F.col('test_failed') == 'accepted_values').count()
    relationship_failures = failed_df.filter(F.col('test_failed') == 'relationships').count()
else:
    unique_failures = not_null_failures = accepted_failures = relationship_failures = 0

# Get total records from the source table
total_records = spark.table(config['table'])
if config['filter']:
    total_records = total_records.filter(config['filter'])
total_count = total_records.count()

passed_count = total_count - failed_count

# Create metrics row
metrics_data = [(
    table_name,
    datetime.now(),
    total_count,
    passed_count,
    failed_count,
    unique_failures + not_null_failures,  # critical_issues
    accepted_failures + relationship_failures,  # error_issues
    0,  # warning_issues
    round(passed_count * 100.0 / total_count, 2) if total_count > 0 else 0.0,  # completeness_rate
    round(passed_count * 100.0 / total_count, 2) if total_count > 0 else 0.0,  # validity_rate
    0.0,  # run_duration_seconds
    f'{{"unique":{unique_failures},"not_null":{not_null_failures},"accepted_values":{accepted_failures},"relationships":{relationship_failures}}}'  # check_details
)]

# Define schema explicitly
metrics_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("check_timestamp", TimestampType(), False),
    StructField("total_records", LongType(), False),
    StructField("passed_records", LongType(), False),
    StructField("failed_records", LongType(), False),
    StructField("critical_issues", LongType(), False),
    StructField("error_issues", LongType(), False),
    StructField("warning_issues", LongType(), False),
    StructField("completeness_rate", DoubleType(), False),
    StructField("validity_rate", DoubleType(), False),
    StructField("run_duration_seconds", DoubleType(), False),
    StructField("check_details", StringType(), True)
])

metrics_df = spark.createDataFrame(metrics_data, metrics_schema)
metrics_df.write.mode('append').saveAsTable('workspace.gold.data_quality_metrics')

print('✅ Metrics logged to gold.data_quality_metrics')

In [0]:
# Quarantine failed records
from pyspark.sql import functions as F

failed_df = spark.table('failed_records')
failed_count = failed_df.count()

if failed_count > 0:
    # Add quarantine metadata
    quarantine_df = failed_df \
        .withColumn('failure_severity', 
                   F.when(F.col('test_failed').isin('unique', 'not_null'), F.lit('CRITICAL'))
                    .otherwise(F.lit('ERROR'))) \
        .withColumn('quarantine_timestamp', F.current_timestamp()) \
        .withColumn('remediation_status', F.lit('PENDING')) \
        .withColumn('remediation_notes', F.lit(None).cast('string')) \
        .drop('test_failed', 'column_name')
    
    # Write to quarantine (append to keep history, allow schema evolution)
    quarantine_df.write.mode('append').option('mergeSchema', 'true').saveAsTable(config['quarantine'])
    
    print(f"⚠️  Quarantined {failed_count} records to {config['quarantine']}")
    
    # Show breakdown
    failed_df.groupBy('test_failed').count().orderBy('test_failed').show()
else:
    print("✅ No records to quarantine - all tests passed!")

In [0]:
%sql
-- Show quality trends over time (last 10 runs)
SELECT 
  check_timestamp,
  total_records,
  passed_records,
  failed_records,
  CONCAT(CAST(completeness_rate AS STRING), '%') as completeness_rate,
  CONCAT(CAST(validity_rate AS STRING), '%') as validity_rate,
  CASE 
    WHEN critical_issues > 0 THEN '🔴 CRITICAL'
    WHEN error_issues > 0 THEN '🟠 ERROR'
    WHEN warning_issues > 0 THEN '🟡 WARNING'
    ELSE '✅ PASS'
  END as status,
  critical_issues,
  error_issues,
  warning_issues
FROM workspace.gold.data_quality_metrics
WHERE table_name = :table_name
ORDER BY check_timestamp DESC
LIMIT 10;